# `05_irr_reliability.ipynb`

This notebook assesses **inter-annotator reliability (IRR)** for the AnnoTinder annotations using
**Krippendorff’s α (ordinal)**. It evaluates agreement on the main outcome dimensions
(*negative sentiment, misinformation, toxicity*) under different experimental conditions.

## Purpose
The goal is to verify that annotations are sufficiently reliable to support downstream
modeling and causal analyses. Reliability is assessed:
- on the **full 5-point ordinal scale** used by annotators, and
- via a **coarsened (“soft”) 3-category ordinal scale** as a robustness check.

All analyses operate on the **final merged analysis dataset** and are performed at the
**item-by-annotator** level.

---

## Inputs
- **Merged analysis dataset** (item × annotator level):
  - `data/final_merged_dataset_for_analysis.parquet`
- **Required columns in the merged dataset**:
  - `unit_id` — unique identifier for the annotated item (tweet)
  - `coder` — annotator identifier
  - `variable` — annotation dimension:
    - `stellingen.sentiment`
    - `stellingen.misinformation`
    - `stellingen.toxic`
  - `value` / `value_scaled` — annotator response (numeric or mappable to 1–5)
  - `source_shown` — whether the source was visible (0 = masked, 1 = shown)
  - `instruction_type` — experimental instruction condition  
    (`no instructions`, `general instructions`, `tailored instructions`)
- **Configuration and utilities**:
  - `config.py` — WebDAV credentials and project root
  - `krippendorff`, `scipy`, `numpy`, `pandas`

---

## What the notebook does

### 1) Prepares ordinal ratings
- Harmonizes annotation responses to a numeric **1–5 ordinal scale**.
- Creates an additional **3-category (“soft”) ordinal scale** for robustness:
  - **1–2 → low/weak** (e.g., low negativity / no misinformation / no toxicity)
  - **3 → neutral**
  - **4–5 → high/strong**

### 2) Builds item × annotator matrices
- Constructs item-by-annotator matrices per construct and experimental subset.
- Excludes items with **fewer than two** annotators **within a subset**.

### 3) Computes Krippendorff’s α
Ordinal α is computed for:
- **Overall**
- **Source visibility** (shown vs. masked)
- **Instruction condition** (none / general / tailored)

### 4) Estimates uncertainty (main)
- Uses **item-level subsample bootstrap** (m-out-of-n, **without replacement**).
- Reports **95% bootstrap percentile confidence intervals** as the primary uncertainty measure.

### 5) Robustness checks
- **BCa confidence intervals** (bias-corrected and accelerated) are computed as a robustness check,
  using K-fold delete-chunk jackknife to estimate acceleration.
- All analyses are repeated on the **soft 3-category scale** to assess robustness to coarsening.
- (Optional) compact “presentation” tables are generated with conditions as columns for quick
  visual comparison.

---

## Outputs
All outputs are written to the Research Drive under `output/tables/`.

### Main results (percentile bootstrap CIs)

**5-point scale**
- LaTeX table:  
  - `output/tables/irr_alpha_table_5pt_percentile.tex`
- CSV summary:  
  - `output/tables/irr_alpha_summary_5pt_percentile.csv`
- Interpretation text:  
  - `output/tables/irr_alpha_interpretation_5pt_percentile.txt`
- Presentation-style wide table (optional):  
  - `output/tables/irr_alpha_table_5pt_percentile_presentation.tex`  
  - `output/tables/irr_alpha_table_5pt_percentile_presentation.csv`

**Soft 3-category scale**
- LaTeX table:  
  - `output/tables/irr_alpha_table_soft3_percentile.tex`
- CSV summary:  
  - `output/tables/irr_alpha_summary_soft3_percentile.csv`
- Interpretation text:  
  - `output/tables/irr_alpha_interpretation_soft3_percentile.txt`
- Presentation-style wide table (optional):  
  - `output/tables/irr_alpha_table_soft3_percentile_presentation.tex`  
  - `output/tables/irr_alpha_table_soft3_percentile_presentation.csv`

---

### Robustness (BCa confidence intervals)

**5-point scale**
- LaTeX table:  
  - `output/tables/irr_alpha_table_5pt_bca_robustness.tex`
- CSV summary:  
  - `output/tables/irr_alpha_summary_5pt_bca_robustness.csv`
- Interpretation text:  
  - `output/tables/irr_alpha_interpretation_5pt_bca_robustness.txt`
- Presentation-style wide table (optional):  
  - `output/tables/irr_alpha_table_5pt_bca_presentation.tex`  
  - `output/tables/irr_alpha_table_5pt_bca_presentation.csv`

**Soft 3-category scale**
- LaTeX table:  
  - `output/tables/irr_alpha_table_soft3_bca_robustness.tex`
- CSV summary:  
  - `output/tables/irr_alpha_summary_soft3_bca_robustness.csv`
- Interpretation text:  
  - `output/tables/irr_alpha_interpretation_soft3_bca_robustness.txt`
- Presentation-style wide table (optional):  
  - `output/tables/irr_alpha_table_soft3_bca_presentation.tex`  
  - `output/tables/irr_alpha_table_soft3_bca_presentation.csv`

---

## Notes
- Krippendorff’s α is computed on the **full sample per subset**.
- Uncertainty is quantified via **item-level resampling**, treating annotated items (tweets) as the
  independent unit.
- Items with fewer than two ratings **within a subset** are excluded by design.
- Rater counts reflect **unique annotators who contribute at least one included rating** (i.e., who
  appear in the item × rater matrix after exclusions).
- Bootstrap diagnostics (e.g., number of valid resamples) are recorded in the summary outputs.


In [1]:
# -*- coding: utf-8 -*-
# ===========================================================
# 05_IRR_ALPHA (Krippendorff α; ordinal)
# MAIN: 95% bootstrap percentile CIs (item-resampling)
# ROBUSTNESS: BCa CIs (jackknife acceleration) exported separately
#
# ALSO: "soft" 3-category agreement (ordinal):
#   1–2 = low/weak (e.g., low negativity / no misinfo / no toxicity)
#   3   = neutral
#   4–5 = high/strong (e.g., strong negativity / misinfo / toxicity)
#
# Exports (Research Drive -> output/tables/):
#   5-point: long main + long BCa + APA overview main + APA overview BCa (+ CSV + TXT)
#   soft3:   long main + long BCa + APA overview main + APA overview BCa (+ CSV + TXT)
#
# Notes:
# - Resampling is at the item (tweet) level.
# - Items with <2 ratings within a subset are excluded.
# - raters_used counts coders appearing in the item×rater matrix (after exclusions).
# - APA overview table includes:
#     (a) grouped header rows ("Source visibility", "Instruction type")
#     (b) Items(n) and Raters(n) rows under the header
#     (c) α [CI] in ONE LINE per cell
# ===========================================================

import numpy as np
import pandas as pd
from scipy import stats
import krippendorff
from datetime import datetime

import config
import rd_utils as rd
from rd_utils import webdav_mkdirs, webdav_upload_bytes

# ---------------------------
# Paths (Research Drive)
# ---------------------------
PROJ       = config.PROJECT_ROOT
DATA_DIR   = f"{PROJ}/data"
TABLES_DIR = f"{PROJ}/output/tables"
INPUT_PARQUET = "final_merged_dataset_for_analysis.parquet"

# ---- 5-point outputs ----
OUT_TEX_MAIN_5   = f"{TABLES_DIR}/irr_alpha_table_5pt_percentile.tex"
OUT_CSV_MAIN_5   = f"{TABLES_DIR}/irr_alpha_summary_5pt_percentile.csv"
OUT_TXT_MAIN_5   = f"{TABLES_DIR}/irr_alpha_interpretation_5pt_percentile.txt"

OUT_TEX_BCA_5    = f"{TABLES_DIR}/irr_alpha_table_5pt_bca_robustness.tex"
OUT_CSV_BCA_5    = f"{TABLES_DIR}/irr_alpha_summary_5pt_bca_robustness.csv"
OUT_TXT_BCA_5    = f"{TABLES_DIR}/irr_alpha_interpretation_5pt_bca_robustness.txt"

OUT_TEX_OVERVIEW_5      = f"{TABLES_DIR}/irr_alpha_overview_5pt_percentile_apa.tex"
OUT_CSV_OVERVIEW_5      = f"{TABLES_DIR}/irr_alpha_overview_5pt_percentile.csv"
OUT_TEX_OVERVIEW_BCA_5  = f"{TABLES_DIR}/irr_alpha_overview_5pt_bca_apa.tex"
OUT_CSV_OVERVIEW_BCA_5  = f"{TABLES_DIR}/irr_alpha_overview_5pt_bca.csv"

# ---- soft 3-category outputs ----
OUT_TEX_MAIN_S3  = f"{TABLES_DIR}/irr_alpha_table_soft3_percentile.tex"
OUT_CSV_MAIN_S3  = f"{TABLES_DIR}/irr_alpha_summary_soft3_percentile.csv"
OUT_TXT_MAIN_S3  = f"{TABLES_DIR}/irr_alpha_interpretation_soft3_percentile.txt"

OUT_TEX_BCA_S3   = f"{TABLES_DIR}/irr_alpha_table_soft3_bca_robustness.tex"
OUT_CSV_BCA_S3   = f"{TABLES_DIR}/irr_alpha_summary_soft3_bca_robustness.csv"
OUT_TXT_BCA_S3   = f"{TABLES_DIR}/irr_alpha_interpretation_soft3_bca_robustness.txt"

OUT_TEX_OVERVIEW_S3      = f"{TABLES_DIR}/irr_alpha_overview_soft3_percentile_apa.tex"
OUT_CSV_OVERVIEW_S3      = f"{TABLES_DIR}/irr_alpha_overview_soft3_percentile.csv"
OUT_TEX_OVERVIEW_BCA_S3  = f"{TABLES_DIR}/irr_alpha_overview_soft3_bca_apa.tex"
OUT_CSV_OVERVIEW_BCA_S3  = f"{TABLES_DIR}/irr_alpha_overview_soft3_bca.csv"

# ---------------------------
# Columns
# ---------------------------
ITEM_COL  = "unit_id"
RATER_COL = "coder"
VAR_COL   = "variable"
SRC_COL   = "source_shown"         # 0/1
INST_COL  = "instruction_type"     # strings

# ---------------------------
# Constructs
# ---------------------------
CONSTRUCTS = {
    "stellingen.sentiment":      "Negative Sentiment",
    "stellingen.misinformation": "Misinformation",
    "stellingen.toxic":          "Toxicity",
}
CONSTRUCT_ORDER = ["Negative Sentiment", "Misinformation", "Toxicity"]

# ---------------------------
# Subsets
# ---------------------------
SUBSET_DISPLAY = {
    "Overall": "Overall",
    "source_shown=1": "Source shown",
    "source_shown=0": "Source masked",
    "instruction_type=no instructions": "No instructions",
    "instruction_type=general instructions": "General instructions",
    "instruction_type=tailored instructions": "Tailored instructions",
}
SUBSET_ORDER = [
    "Overall",
    "Source shown", "Source masked",
    "No instructions", "General instructions", "Tailored instructions",
]
SUBSET_SPECS = [
    ("Overall", None, None),
    ("source_shown=1", SRC_COL, 1),
    ("source_shown=0", SRC_COL, 0),
    ("instruction_type=no instructions",       INST_COL, "no instructions"),
    ("instruction_type=general instructions",  INST_COL, "general instructions"),
    ("instruction_type=tailored instructions", INST_COL, "tailored instructions"),
]

# ---------------------------
# Uncertainty config
# ---------------------------
RESAMPLE_SCHEME  = "subsample"  # item resampling without replacement
M_RATIO          = 0.80
B                = 1000
SEED             = 123
K_FOLDS          = 10  # for BCa acceleration via delete-chunk jackknife

# ===========================================================
# WebDAV helpers
# ===========================================================
def _parent_dir(path: str) -> str:
    parts = path.rsplit("/", 1)
    return parts[0] if len(parts) == 2 else ""

def upload_text(rel_path: str, text: str, content_type: str = "text/plain; charset=utf-8"):
    webdav_mkdirs(_parent_dir(rel_path))
    webdav_upload_bytes(rel_path, text.encode("utf-8"), content_type=content_type)
    print(f"✅ Uploaded {rel_path}")

def upload_bytes(rel_path: str, b: bytes, content_type: str):
    webdav_mkdirs(_parent_dir(rel_path))
    webdav_upload_bytes(rel_path, b, content_type=content_type)
    print(f"✅ Uploaded {rel_path}")

# ===========================================================
# Rating preparation
# ===========================================================
def ensure_rating_1_5(
    df: pd.DataFrame,
    prefer=("value_scaled", "value"),
    label_cols=("value", "answer", "response", "value_label", "code", "rating"),
) -> pd.DataFrame:
    """
    Create numeric 1..5 column '_rating5' from:
      - numeric 1..5
      - numeric 0..1 (mapped to 1..5)
      - Dutch Likert labels (mapped)
    """
    df = df.copy()

    def try_numeric(col):
        if col not in df.columns:
            return None, None
        x = pd.to_numeric(df[col], errors="coerce")
        if (~x.isna()).mean() < 0.80:
            return None, None
        lo, hi = float(x.min()), float(x.max())

        if 1 - 1e-6 <= lo and hi <= 5 + 1e-6:
            return x.round().clip(1, 5), f"{col} (already 1–5)"
        if -1e-6 <= lo and hi <= 1 + 1e-6:
            y = (1 + 4 * x).round().clip(1, 5)
            return y, f"{col} (0–1 → 1–5)"
        return None, None

    for col in prefer:
        y, msg = try_numeric(col)
        if y is not None:
            df["_rating5"] = y
            print(f"[rating] Using {msg}. valid={df['_rating5'].notna().mean():.3f} "
                  f"min={df['_rating5'].min()} max={df['_rating5'].max()}")
            return df

    LIKERT_MAP_NL = {
        "Helemaal niet waar":     1,
        "Gedeeltelijk niet waar": 2,
        "Neutraal":               3,
        "Gedeeltelijk waar":      4,
        "Helemaal waar":          5,
    }
    for col in label_cols:
        if col in df.columns and df[col].notna().any():
            s = df[col].astype(str).str.strip()
            m = s.map(LIKERT_MAP_NL)
            if m.notna().mean() >= 0.80:
                df["_rating5"] = m
                print(f"[rating] Using {col} (mapped labels). valid={m.notna().mean():.3f}")
                return df

    raise ValueError("No usable rating column found (need numeric 1–5, numeric 0–1, or mappable labels).")

def apply_scale(df: pd.DataFrame, scale: str) -> pd.DataFrame:
    """
    Adds '_rating_work' depending on scale.
    scale:
      - '5pt': use 1..5
      - 'soft3': 1–2 low, 3 neutral, 4–5 high (ordinal 3-category)
    """
    df = df.copy()
    df["_rating_work"] = df["_rating5"]

    if scale == "5pt":
        return df

    if scale == "soft3":
        x = pd.to_numeric(df["_rating5"], errors="coerce")
        y = pd.Series(np.nan, index=df.index, dtype=float)
        y[(x >= 1) & (x <= 2)] = 1
        y[x == 3] = 2
        y[(x >= 4) & (x <= 5)] = 3
        df["_rating_work"] = y
        return df

    raise ValueError("scale must be '5pt' or 'soft3'.")

# ===========================================================
# Matrix building
# ===========================================================
def pivot_items_raters(dfc: pd.DataFrame, rating_col="_rating_work", min_k=2):
    """
    Build items×raters matrix; drop items with < min_k raters in subset.
    Returns (M, items_used, raters_used).
    """
    if dfc.empty:
        return None, [], 0

    use = dfc[[ITEM_COL, RATER_COL, rating_col]].copy()
    use[rating_col] = pd.to_numeric(use[rating_col], errors="coerce")
    use = use.dropna(subset=[rating_col])
    if use.empty:
        return None, [], 0

    # collapse duplicates per item×coder
    use = use.groupby([ITEM_COL, RATER_COL], as_index=False)[rating_col].mean()

    # keep items with >= min_k unique raters
    k = use.groupby(ITEM_COL)[RATER_COL].nunique()
    keep_items = k[k >= min_k].index
    use = use[use[ITEM_COL].isin(keep_items)]
    if use.empty:
        return None, [], 0

    pivot = use.pivot(index=ITEM_COL, columns=RATER_COL, values=rating_col)
    M = pivot.to_numpy()
    items_used = list(pivot.index)
    raters_used = int(pivot.columns.size)
    return M, items_used, raters_used

# ===========================================================
# Alpha + uncertainty
# ===========================================================
def alpha_ordinal(M: np.ndarray) -> float:
    if M is None or M.shape[0] == 0:
        return np.nan
    return float(krippendorff.alpha(M, level_of_measurement="ordinal"))

def bootstrap_alphas(M, B=B, seed=SEED, scheme=RESAMPLE_SCHEME, m_ratio=M_RATIO):
    n = M.shape[0]
    rng = np.random.default_rng(seed)
    boots = np.empty(B, dtype=float)

    if scheme == "subsample":
        m = max(2, int(round(m_ratio * n)))
        for b in range(B):
            sel = rng.choice(n, size=m, replace=False)
            try:
                boots[b] = krippendorff.alpha(M[sel], level_of_measurement="ordinal")
            except Exception:
                boots[b] = np.nan
    elif scheme == "bootstrap":
        idx = np.arange(n)
        for b in range(B):
            sel = rng.choice(idx, size=n, replace=True)
            try:
                boots[b] = krippendorff.alpha(M[sel], level_of_measurement="ordinal")
            except Exception:
                boots[b] = np.nan
    else:
        raise ValueError("scheme must be 'subsample' or 'bootstrap'")

    return boots

def percentile_ci(boots, alpha=0.05):
    lo, hi = np.nanpercentile(boots, [100 * alpha / 2, 100 * (1 - alpha / 2)])
    return float(lo), float(hi)

def jackknife_alphas_kfold(M, K=K_FOLDS, seed=SEED):
    n = M.shape[0]
    K = int(min(max(2, K), n))
    rng = np.random.default_rng(seed)
    perm = rng.permutation(n)
    folds = np.array_split(perm, K)

    jvals = np.empty(K, dtype=float)
    for i, fold in enumerate(folds):
        keep = np.setdiff1d(np.arange(n), fold, assume_unique=False)
        if keep.size < 2:
            jvals[i] = np.nan
            continue
        try:
            jvals[i] = krippendorff.alpha(M[keep], level_of_measurement="ordinal")
        except Exception:
            jvals[i] = np.nan
    return jvals

def bca_ci(a_hat, boots, jacks, alpha=0.05):
    boots = np.asarray(boots, float)
    jacks = np.asarray(jacks, float)

    # bias-correction
    prop_less = np.nanmean(boots < a_hat)
    eps = 1e-10
    prop_less = min(max(prop_less, eps), 1 - eps)
    z0 = stats.norm.ppf(prop_less)

    # acceleration (jackknife)
    j_mean = np.nanmean(jacks)
    num = np.nansum((j_mean - jacks) ** 3)
    den = 6.0 * (np.nansum((j_mean - jacks) ** 2) ** 1.5)
    if (not np.isfinite(den)) or den == 0.0:
        acc = 0.0
    else:
        acc = float(num / den)
        if not np.isfinite(acc):
            acc = 0.0

    def pct(q):
        z = stats.norm.ppf(q)
        denom = 1 - acc * (z0 + z)
        z_adj = z0 + (z0 + z) / denom if denom != 0 else z0
        return stats.norm.cdf(z_adj)

    ql = np.nanpercentile(boots, 100 * pct(alpha / 2))
    qh = np.nanpercentile(boots, 100 * pct(1 - alpha / 2))
    return float(ql), float(qh)

def alpha_point_and_cis(M, B=B, seed=SEED):
    if M is None or M.shape[0] == 0:
        return np.nan, (np.nan, np.nan), (np.nan, np.nan), {"boot_valid": 0, "boot_total": B}

    a_hat = alpha_ordinal(M)
    boots = bootstrap_alphas(M, B=B, seed=seed, scheme=RESAMPLE_SCHEME, m_ratio=M_RATIO)
    pct_low, pct_high = percentile_ci(boots)

    try:
        jacks = jackknife_alphas_kfold(M, K=K_FOLDS, seed=seed)
        bca_low, bca_high = bca_ci(a_hat, boots, jacks, alpha=0.05)
    except Exception:
        bca_low, bca_high = np.nan, np.nan

    diag = {"boot_valid": int(np.isfinite(boots).sum()), "boot_total": int(B)}
    return a_hat, (pct_low, pct_high), (bca_low, bca_high), diag

# ===========================================================
# Summary (long)
# ===========================================================
def run_irr_summary(df_annot: pd.DataFrame, scale: str) -> pd.DataFrame:
    df = ensure_rating_1_5(df_annot, prefer=("value_scaled", "value"))
    df = apply_scale(df, scale=scale)

    rows = []
    for var_key, display in CONSTRUCTS.items():
        base = df.loc[df[VAR_COL] == var_key].copy()

        for code, col, val in SUBSET_SPECS:
            block = base if col is None else base.loc[base[col] == val]

            M, items_used, raters_used = pivot_items_raters(block, rating_col="_rating_work", min_k=2)
            a_hat, (pct_low, pct_high), (bca_low, bca_high), diag = alpha_point_and_cis(M, B=B, seed=SEED)

            rows.append({
                "construct": display,
                "subset_code": code,
                "subset_disp": SUBSET_DISPLAY.get(code, code),
                "alpha": a_hat,
                "pct_low": pct_low,
                "pct_high": pct_high,
                "bca_low": bca_low,
                "bca_high": bca_high,
                "items": int(len(items_used)),
                "raters_used": int(raters_used),
                "boot_valid": int(diag["boot_valid"]),
                "boot_total": int(diag["boot_total"]),
                "scale": scale,
            })

    out = pd.DataFrame(rows)
    out["construct"] = pd.Categorical(out["construct"], categories=CONSTRUCT_ORDER, ordered=True)
    out["subset_disp"] = pd.Categorical(out["subset_disp"], categories=SUBSET_ORDER, ordered=True)
    out = out.sort_values(["construct", "subset_disp"]).reset_index(drop=True)
    return out

# ===========================================================
# Long LaTeX (subset-by-construct)
# ===========================================================
def _fmt_int_ltx(n: int) -> str:
    return f"{int(n):,}".replace(",", "{,}")

def latex_table_long(summary_df: pd.DataFrame, ci_kind: str, label: str, caption: str, note: str) -> str:
    assert ci_kind in ("percentile", "bca")
    lo_col, hi_col = ("pct_low", "pct_high") if ci_kind == "percentile" else ("bca_low", "bca_high")

    lines = [
        r"\begin{table}[tbp]",
        r"\centering",
        rf"\caption{{{caption}}}",
        rf"\label{{{label}}}",
        r"\begin{tabularx}{\linewidth}{l l c c c c}",
        r"\toprule",
        r"\textbf{Construct} & \textbf{Subset} & $\boldsymbol{\alpha}$ & \textbf{95\% CI} & \textbf{Items} & \textbf{Raters} \\",
        r"\midrule",
    ]

    for _, r in summary_df.iterrows():
        a   = "NA" if pd.isna(r["alpha"]) else f"{r['alpha']:.3f}"
        cil = "NA" if pd.isna(r[lo_col]) else f"{r[lo_col]:.3f}"
        cih = "NA" if pd.isna(r[hi_col]) else f"{r[hi_col]:.3f}"
        items  = _fmt_int_ltx(r["items"])
        raters = _fmt_int_ltx(r["raters_used"])
        lines.append(f"{r['construct']} & {r['subset_disp']} & {a} & [{cil}, {cih}] & {items} & {raters} \\\\")

    lines += [
        r"\bottomrule",
        r"\end{tabularx}",
        note,
        r"\end{table}",
    ]
    return "\n".join(lines)

# ===========================================================
# Interpretation text
# ===========================================================
def interpretation_text(summary_df: pd.DataFrame, ci_kind: str, scale_label: str) -> str:
    ts = datetime.now().strftime("%Y-%m-%d %H:%M")
    if ci_kind == "percentile":
        head = (
            f"Inter-annotator agreement (Krippendorff’s alpha, ordinal; {scale_label}). "
            f"Bootstrap percentile 95% CIs. Generated: {ts}\n"
            f"Resampling: subsample without replacement (m={M_RATIO}·n), B={B}\n"
        )
        lo_col, hi_col = "pct_low", "pct_high"
    else:
        head = (
            f"Robustness check: Krippendorff’s alpha with BCa 95% CIs (ordinal; {scale_label}). Generated: {ts}\n"
            f"Resampling: subsample without replacement (m={M_RATIO}·n), B={B}; "
            f"BCa acceleration via K-fold delete-chunk jackknife (K={K_FOLDS}).\n"
        )
        lo_col, hi_col = "bca_low", "bca_high"

    parts = [head, ""]
    for cons in CONSTRUCT_ORDER:
        sub = summary_df.loc[summary_df["construct"] == cons]
        if sub.empty:
            continue

        parts.append(cons + ":")
        for subset in SUBSET_ORDER:
            row = sub.loc[sub["subset_disp"] == subset]
            if row.empty:
                continue
            row = row.iloc[0]
            parts.append(
                f"  - {subset}: α={row['alpha']:.3f} [{row[lo_col]:.3f}, {row[hi_col]:.3f}] "
                f"(items={int(row['items'])}, raters={int(row['raters_used'])}, "
                f"boot_valid={int(row['boot_valid'])}/{int(row['boot_total'])})"
            )
        parts.append("")
    return "\n".join(parts)

# ===========================================================
# Overview (wide) tables with counts
# ===========================================================
def build_overview_df(summary_df: pd.DataFrame, ci_kind: str) -> pd.DataFrame:
    """
    Wide table:
      rows = construct
      cols = SUBSET_ORDER
      cell = alpha [lo, hi]
    """
    assert ci_kind in ("percentile", "bca")
    lo_col, hi_col = ("pct_low", "pct_high") if ci_kind == "percentile" else ("bca_low", "bca_high")

    df = summary_df.copy()

    def cell(r):
        if pd.isna(r["alpha"]) or pd.isna(r[lo_col]) or pd.isna(r[hi_col]):
            return "NA"
        return f"{r['alpha']:.3f} [{r[lo_col]:.3f}, {r[hi_col]:.3f}]"

    df["cell"] = df.apply(cell, axis=1)
    df = df[df["subset_disp"].isin(SUBSET_ORDER)].copy()
    df["subset_disp"] = pd.Categorical(df["subset_disp"], categories=SUBSET_ORDER, ordered=True)

    wide = (
        df.pivot_table(index="construct", columns="subset_disp", values="cell", aggfunc="first")
          .reindex(index=CONSTRUCT_ORDER)
          .reset_index()
    )
    wide.columns.name = None
    return wide

def overview_counts(summary_df: pd.DataFrame) -> pd.DataFrame:
    """
    Returns subset-level counts for Items(n) and Raters(n).
    We use max across constructs (safe when identical; still works if not).
    """
    tmp = (summary_df
           .groupby("subset_disp", as_index=False)
           .agg(items=("items", "max"),
                raters_used=("raters_used", "max")))
    tmp["subset_disp"] = pd.Categorical(tmp["subset_disp"], categories=SUBSET_ORDER, ordered=True)
    tmp = tmp.sort_values("subset_disp").reset_index(drop=True)
    return tmp

# ===========================================================
# APA-style overview LaTeX with grouped headers + N rows
# ===========================================================
def latex_overview_table_apa_grouped(
    wide_df: pd.DataFrame,
    counts_df: pd.DataFrame,
    label: str,
    caption: str,
    note: str,
) -> str:
    """
    APA-ish overview table:
      - Grouped header row: Source visibility, Instruction type
      - Second header row: levels (shown/masked, etc.)
      - Two rows under headers: Items(n) and Raters(n)
      - Body rows: construct with alpha [CI] in one line per cell

    Requires LaTeX packages:
      booktabs, tabularx, threeparttable, array
    """
    cols = ["construct"] + SUBSET_ORDER
    df = wide_df.copy()
    for c in cols:
        if c not in df.columns:
            df[c] = "NA"
    df = df[cols]

    counts = counts_df.set_index("subset_disp")

    def fmt_n(x):
        return f"{int(x):,}".replace(",", "{,}")

    def n_items(col):
        return "NA" if col not in counts.index else fmt_n(counts.loc[col, "items"])

    def n_raters(col):
        return "NA" if col not in counts.index else fmt_n(counts.loc[col, "raters_used"])

    # Short labels for fit (only in column headers)
    col_hdr = {
        "Overall": "Overall",
        "Source shown": "Shown",
        "Source masked": "Masked",
        "No instructions": "None",
        "General instructions": "General",
        "Tailored instructions": "Tailored",
    }

    lines = [
        r"\begin{table}[tbp]",
        r"\centering",
        r"\begin{threeparttable}",
        rf"\caption{{{caption}}}",
        rf"\label{{{label}}}",
        r"\footnotesize",
        r"\setlength{\tabcolsep}{3.5pt}",
        r"\renewcommand{\arraystretch}{1.15}",
        r"\begin{tabularx}{\linewidth}{>{\raggedright\arraybackslash}l *{6}{>{\centering\arraybackslash}X}}",
        r"\toprule",
        # Group header row
        r"\textbf{Construct} & \textbf{Overall} & \multicolumn{2}{c}{\textbf{Source visibility}} & \multicolumn{3}{c}{\textbf{Instruction type}} \\",
        # cmidrules (booktabs)
        r"\cmidrule(lr){3-4}\cmidrule(lr){5-7}",
        # Level header row
        r" &  & \textbf{" + col_hdr["Source shown"] + r"} & \textbf{" + col_hdr["Source masked"] + r"} & "
        r"\textbf{" + col_hdr["No instructions"] + r"} & \textbf{" + col_hdr["General instructions"] + r"} & \textbf{" + col_hdr["Tailored instructions"] + r"} \\",
        r"\midrule",
        # N rows (counts)
        r"\textit{Items} ($n$) & " + n_items("Overall") + " & " + n_items("Source shown") + " & " + n_items("Source masked")
        + " & " + n_items("No instructions") + " & " + n_items("General instructions") + " & " + n_items("Tailored instructions") + r" \\",
        r"\textit{Raters} ($n$) & " + n_raters("Overall") + " & " + n_raters("Source shown") + " & " + n_raters("Source masked")
        + " & " + n_raters("No instructions") + " & " + n_raters("General instructions") + " & " + n_raters("Tailored instructions") + r" \\",
        r"\addlinespace[0.35em]",
    ]

    for _, r in df.iterrows():
        lines.append(
            f"\\textbf{{{r['construct']}}} & "
            f"{r['Overall']} & {r['Source shown']} & {r['Source masked']} & "
            f"{r['No instructions']} & {r['General instructions']} & {r['Tailored instructions']} \\\\"
        )

    lines += [
        r"\bottomrule",
        r"\end{tabularx}",
        r"\begin{tablenotes}[flushleft]",
        r"\footnotesize",
        rf"\item {note}",
        r"\end{tablenotes}",
        r"\end{threeparttable}",
        r"\end{table}",
    ]
    return "\n".join(lines)

# ===========================================================
# Export per scale
# ===========================================================
def export_scale(
    summary: pd.DataFrame,
    scale: str,
    out_tex_main: str, out_csv_main: str, out_txt_main: str,
    out_tex_bca: str, out_csv_bca: str, out_txt_bca: str,
    out_tex_over: str, out_csv_over: str,
    out_tex_over_bca: str, out_csv_over_bca: str,
):
    webdav_mkdirs(TABLES_DIR)

    # ---- long MAIN (percentile) ----
    main_cols = [
        "construct", "subset_code", "subset_disp", "alpha",
        "pct_low", "pct_high", "items", "raters_used", "boot_valid", "boot_total"
    ]
    upload_bytes(out_csv_main, summary[main_cols].to_csv(index=False).encode("utf-8"),
                 content_type="text/csv; charset=utf-8")

    if scale == "5pt":
        cap_main = "Inter-annotator agreement (Krippendorff's $\\alpha$, ordinal; 5-point scale)."
        lab_main = "tab:irr_alpha_5pt_percentile"
        scale_note = ""
    else:
        cap_main = "Inter-annotator agreement (Krippendorff's $\\alpha$, ordinal; coarsened 3-category scale)."
        lab_main = "tab:irr_alpha_soft3_percentile"
        scale_note = " Coarsened scale: 1--2 (low), 3 (neutral), 4--5 (high), treated as ordinal."

    note_main = (
        r"\par\vspace{0.5ex}\parbox{\linewidth}{\footnotesize\emph{Note}. "
        r"Values are Krippendorff’s $\alpha$ (ordinal)." + scale_note +
        r" Confidence intervals are 95\% bootstrap percentile intervals based on item resampling "
        r"(subsample without replacement; $m=0.80\cdot n$, $B=1000$). "
        r"Items with fewer than two ratings within a subset are excluded.}"
    )
    upload_text(out_tex_main, latex_table_long(summary, "percentile", lab_main, cap_main, note_main),
                content_type="text/x-tex; charset=utf-8")
    upload_text(out_txt_main, interpretation_text(summary, "percentile", "5-point" if scale == "5pt" else "soft3"),
                content_type="text/plain; charset=utf-8")

    # ---- long BCa ----
    bca_cols = [
        "construct", "subset_code", "subset_disp", "alpha",
        "bca_low", "bca_high", "items", "raters_used", "boot_valid", "boot_total"
    ]
    upload_bytes(out_csv_bca, summary[bca_cols].to_csv(index=False).encode("utf-8"),
                 content_type="text/csv; charset=utf-8")

    if scale == "5pt":
        cap_bca = "Robustness check: inter-annotator agreement with BCa confidence intervals (5-point scale)."
        lab_bca = "tab:irr_alpha_5pt_bca"
    else:
        cap_bca = "Robustness check: inter-annotator agreement with BCa confidence intervals (coarsened 3-category scale)."
        lab_bca = "tab:irr_alpha_soft3_bca"

    note_bca = (
        r"\par\vspace{0.5ex}\parbox{\linewidth}{\footnotesize\emph{Note}. "
        r"BCa confidence intervals use the same bootstrap draws as the main analysis, with bias-correction and "
        r"jackknife-based acceleration estimated via K-fold delete-chunk jackknife ($K=10$).}"
    )
    upload_text(out_tex_bca, latex_table_long(summary, "bca", lab_bca, cap_bca, note_bca),
                content_type="text/x-tex; charset=utf-8")
    upload_text(out_txt_bca, interpretation_text(summary, "bca", "5-point" if scale == "5pt" else "soft3"),
                content_type="text/plain; charset=utf-8")

    # ---- overview MAIN (percentile) ----
    wide_pct = build_overview_df(summary, "percentile")
    cnt = overview_counts(summary)
    upload_bytes(out_csv_over, wide_pct.to_csv(index=False).encode("utf-8"),
                 content_type="text/csv; charset=utf-8")

    if scale == "5pt":
        cap_over = "Inter-annotator agreement overview (Krippendorff's $\\alpha$, ordinal; 5-point scale)."
        lab_over = "tab:irr_alpha_overview_5pt_percentile"
        note_over = (
            r"\emph{Note}. Cells report $\alpha$ with 95\% bootstrap percentile confidence intervals from item resampling "
            r"(subsample without replacement; $m=0.80\cdot n$, $B=1000$)."
        )
    else:
        cap_over = "Inter-annotator agreement overview (Krippendorff's $\\alpha$, ordinal; coarsened 3-category scale)."
        lab_over = "tab:irr_alpha_overview_soft3_percentile"
        note_over = (
            r"\emph{Note}. Coarsened scale: 1--2 (low), 3 (neutral), 4--5 (high). "
            r"Cells report $\alpha$ with 95\% bootstrap percentile confidence intervals from item resampling "
            r"(subsample without replacement; $m=0.80\cdot n$, $B=1000$)."
        )

    upload_text(
        out_tex_over,
        latex_overview_table_apa_grouped(wide_pct, cnt, lab_over, cap_over, note_over),
        content_type="text/x-tex; charset=utf-8",
    )

    # ---- overview BCa ----
    wide_bca = build_overview_df(summary, "bca")
    upload_bytes(out_csv_over_bca, wide_bca.to_csv(index=False).encode("utf-8"),
                 content_type="text/csv; charset=utf-8")

    if scale == "5pt":
        cap_over_bca = "Robustness overview: inter-annotator agreement with BCa confidence intervals (5-point scale)."
        lab_over_bca = "tab:irr_alpha_overview_5pt_bca"
    else:
        cap_over_bca = "Robustness overview: inter-annotator agreement with BCa confidence intervals (coarsened 3-category scale)."
        lab_over_bca = "tab:irr_alpha_overview_soft3_bca"

    note_over_bca = (
        r"\emph{Note}. Cells report $\alpha$ with BCa 95\% confidence intervals. "
        r"Acceleration is estimated via K-fold delete-chunk jackknife ($K=10$)."
    )

    upload_text(
        out_tex_over_bca,
        latex_overview_table_apa_grouped(wide_bca, cnt, lab_over_bca, cap_over_bca, note_over_bca),
        content_type="text/x-tex; charset=utf-8",
    )

# ===========================================================
# Run
# ===========================================================
if __name__ == "__main__":
    print("[Load] Reading merged dataset (parquet) from Research Drive…")
    df_annot = rd.read_parquet(f"{DATA_DIR}/{INPUT_PARQUET}")
    print(f"[Load] rows={df_annot.shape[0]:,} | cols={df_annot.shape[1]:,}")

    # Sanity: required columns
    required = {ITEM_COL, RATER_COL, VAR_COL, SRC_COL, INST_COL}
    missing = sorted(required - set(df_annot.columns))
    if missing:
        raise KeyError(f"Missing required columns: {missing}")

    if not (("value_scaled" in df_annot.columns) or ("value" in df_annot.columns)):
        raise KeyError("Need rating column: 'value_scaled' or 'value' (or adjust ensure_rating_1_5).")

    webdav_mkdirs(TABLES_DIR)

    # ---------------------------
    # 5-point
    # ---------------------------
    print("[Compute] 5-point: α + percentile CIs (main) + BCa (robustness)…")
    summary_5 = run_irr_summary(df_annot, scale="5pt")
    export_scale(
        summary=summary_5,
        scale="5pt",
        out_tex_main=OUT_TEX_MAIN_5, out_csv_main=OUT_CSV_MAIN_5, out_txt_main=OUT_TXT_MAIN_5,
        out_tex_bca=OUT_TEX_BCA_5, out_csv_bca=OUT_CSV_BCA_5, out_txt_bca=OUT_TXT_BCA_5,
        out_tex_over=OUT_TEX_OVERVIEW_5, out_csv_over=OUT_CSV_OVERVIEW_5,
        out_tex_over_bca=OUT_TEX_OVERVIEW_BCA_5, out_csv_over_bca=OUT_CSV_OVERVIEW_BCA_5,
    )

    # ---------------------------
    # soft3
    # ---------------------------
    print("[Compute] soft3: α + percentile CIs (main) + BCa (robustness)…")
    summary_s3 = run_irr_summary(df_annot, scale="soft3")
    export_scale(
        summary=summary_s3,
        scale="soft3",
        out_tex_main=OUT_TEX_MAIN_S3, out_csv_main=OUT_CSV_MAIN_S3, out_txt_main=OUT_TXT_MAIN_S3,
        out_tex_bca=OUT_TEX_BCA_S3, out_csv_bca=OUT_CSV_BCA_S3, out_txt_bca=OUT_TXT_BCA_S3,
        out_tex_over=OUT_TEX_OVERVIEW_S3, out_csv_over=OUT_CSV_OVERVIEW_S3,
        out_tex_over_bca=OUT_TEX_OVERVIEW_BCA_S3, out_csv_over_bca=OUT_CSV_OVERVIEW_BCA_S3,
    )

    print("\n[Done] Wrote 5pt + soft3 tables (long + APA overview; main + BCa robustness) to Research Drive.")


[Load] Reading merged dataset (parquet) from Research Drive…
[Load] rows=153,674 | cols=203
[Compute] 5-point: α + percentile CIs (main) + BCa (robustness)…
[rating] Using value_scaled (already 1–5). valid=0.906 min=1.0 max=5.0
✅ Uploaded ASCOR-FMG-4394-AnNoBias (Projectfolder)/output/tables/irr_alpha_summary_5pt_percentile.csv
✅ Uploaded ASCOR-FMG-4394-AnNoBias (Projectfolder)/output/tables/irr_alpha_table_5pt_percentile.tex
✅ Uploaded ASCOR-FMG-4394-AnNoBias (Projectfolder)/output/tables/irr_alpha_interpretation_5pt_percentile.txt
✅ Uploaded ASCOR-FMG-4394-AnNoBias (Projectfolder)/output/tables/irr_alpha_summary_5pt_bca_robustness.csv
✅ Uploaded ASCOR-FMG-4394-AnNoBias (Projectfolder)/output/tables/irr_alpha_table_5pt_bca_robustness.tex
✅ Uploaded ASCOR-FMG-4394-AnNoBias (Projectfolder)/output/tables/irr_alpha_interpretation_5pt_bca_robustness.txt


/tmp/ipykernel_699031/1487197040.py:502: FutureWarning: The default value of observed=False is deprecated and will change to observed=True in a future version of pandas. Specify observed=False to silence this warning and retain the current behavior
  df.pivot_table(index="construct", columns="subset_disp", values="cell", aggfunc="first")
/tmp/ipykernel_699031/1487197040.py:514: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  tmp = (summary_df


✅ Uploaded ASCOR-FMG-4394-AnNoBias (Projectfolder)/output/tables/irr_alpha_overview_5pt_percentile.csv
✅ Uploaded ASCOR-FMG-4394-AnNoBias (Projectfolder)/output/tables/irr_alpha_overview_5pt_percentile_apa.tex


/tmp/ipykernel_699031/1487197040.py:502: FutureWarning: The default value of observed=False is deprecated and will change to observed=True in a future version of pandas. Specify observed=False to silence this warning and retain the current behavior
  df.pivot_table(index="construct", columns="subset_disp", values="cell", aggfunc="first")


✅ Uploaded ASCOR-FMG-4394-AnNoBias (Projectfolder)/output/tables/irr_alpha_overview_5pt_bca.csv
✅ Uploaded ASCOR-FMG-4394-AnNoBias (Projectfolder)/output/tables/irr_alpha_overview_5pt_bca_apa.tex
[Compute] soft3: α + percentile CIs (main) + BCa (robustness)…
[rating] Using value_scaled (already 1–5). valid=0.906 min=1.0 max=5.0
✅ Uploaded ASCOR-FMG-4394-AnNoBias (Projectfolder)/output/tables/irr_alpha_summary_soft3_percentile.csv
✅ Uploaded ASCOR-FMG-4394-AnNoBias (Projectfolder)/output/tables/irr_alpha_table_soft3_percentile.tex
✅ Uploaded ASCOR-FMG-4394-AnNoBias (Projectfolder)/output/tables/irr_alpha_interpretation_soft3_percentile.txt
✅ Uploaded ASCOR-FMG-4394-AnNoBias (Projectfolder)/output/tables/irr_alpha_summary_soft3_bca_robustness.csv
✅ Uploaded ASCOR-FMG-4394-AnNoBias (Projectfolder)/output/tables/irr_alpha_table_soft3_bca_robustness.tex
✅ Uploaded ASCOR-FMG-4394-AnNoBias (Projectfolder)/output/tables/irr_alpha_interpretation_soft3_bca_robustness.txt


/tmp/ipykernel_699031/1487197040.py:502: FutureWarning: The default value of observed=False is deprecated and will change to observed=True in a future version of pandas. Specify observed=False to silence this warning and retain the current behavior
  df.pivot_table(index="construct", columns="subset_disp", values="cell", aggfunc="first")
/tmp/ipykernel_699031/1487197040.py:514: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  tmp = (summary_df


✅ Uploaded ASCOR-FMG-4394-AnNoBias (Projectfolder)/output/tables/irr_alpha_overview_soft3_percentile.csv
✅ Uploaded ASCOR-FMG-4394-AnNoBias (Projectfolder)/output/tables/irr_alpha_overview_soft3_percentile_apa.tex


/tmp/ipykernel_699031/1487197040.py:502: FutureWarning: The default value of observed=False is deprecated and will change to observed=True in a future version of pandas. Specify observed=False to silence this warning and retain the current behavior
  df.pivot_table(index="construct", columns="subset_disp", values="cell", aggfunc="first")


✅ Uploaded ASCOR-FMG-4394-AnNoBias (Projectfolder)/output/tables/irr_alpha_overview_soft3_bca.csv
✅ Uploaded ASCOR-FMG-4394-AnNoBias (Projectfolder)/output/tables/irr_alpha_overview_soft3_bca_apa.tex

[Done] Wrote 5pt + soft3 tables (long + APA overview; main + BCa robustness) to Research Drive.
